In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

print("Final project validation started.")

# Project directories
project_dir = Path("..")
data_dir = project_dir / "data"
processed_dir = data_dir / "processed"
models_dir = project_dir / "models"

print("\nProject directory:")
print(project_dir.resolve())

print("\nModels directory:")
print(models_dir.resolve())

print("\nProcessed data directory:")
print(processed_dir.resolve())

Final project validation started.

Project directory:
C:\Users\hsv89\cyber_ml_project

Models directory:
C:\Users\hsv89\cyber_ml_project\models

Processed data directory:
C:\Users\hsv89\cyber_ml_project\data\processed


In [2]:
# Required files for the final intrusion detection pipeline

required_files = {
    "Final XGBoost Model":
        models_dir / "final_xgboost_intrusion_detector.joblib",

    "Deployment Configuration":
        models_dir / "deployment_config.joblib",

    "Preprocessor":
        processed_dir / "preprocessor.joblib",

    "Scaler":
        processed_dir / "scaler.joblib",

    "Final Model Report":
        models_dir / "final_model_report.csv",

    "Test Features":
        processed_dir / "X_test_scaled.npy",

    "Test Labels":
        processed_dir / "y_test.npy"
}

print("FINAL PROJECT FILE CHECK")
print("=" * 55)

all_files_exist = True

for name, path in required_files.items():

    exists = path.exists()

    status = "FOUND" if exists else "MISSING"

    print(f"{name:28} : {status}")

    if not exists:
        all_files_exist = False

print("=" * 55)

if all_files_exist:
    print("All required project files were found!")
else:
    print("WARNING: One or more required files are missing.")

FINAL PROJECT FILE CHECK
Final XGBoost Model          : FOUND
Deployment Configuration     : FOUND
Preprocessor                 : FOUND
Scaler                       : FOUND
Final Model Report           : FOUND
Test Features                : FOUND
Test Labels                  : FOUND
All required project files were found!


In [3]:
# Load all saved pipeline components

model = joblib.load(
    models_dir / "final_xgboost_intrusion_detector.joblib"
)

config = joblib.load(
    models_dir / "deployment_config.joblib"
)

preprocessor = joblib.load(
    processed_dir / "preprocessor.joblib"
)

scaler = joblib.load(
    processed_dir / "scaler.joblib"
)

X_test = np.load(
    processed_dir / "X_test_scaled.npy"
)

y_test = np.load(
    processed_dir / "y_test.npy"
)

final_report = pd.read_csv(
    models_dir / "final_model_report.csv"
)

print("All pipeline components loaded successfully!")

print("\nModel:")
print(type(model))

print("\nPreprocessor:")
print(type(preprocessor))

print("\nScaler:")
print(type(scaler))

print("\nTest data:")
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nDeployment configuration:")
print(config)

All pipeline components loaded successfully!

Model:
<class 'xgboost.sklearn.XGBClassifier'>

Preprocessor:
<class 'sklearn.compose._column_transformer.ColumnTransformer'>

Scaler:
<class 'sklearn.preprocessing._data.StandardScaler'>

Test data:
X_test shape: (82332, 194)
y_test shape: (82332,)

Deployment configuration:
{'model_name': 'final_xgboost_intrusion_detector', 'threshold': 0.5, 'input_features': 194, 'positive_class': 1, 'negative_class': 0}


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Generate probabilities using the freshly loaded model
test_probabilities = model.predict_proba(X_test)[:, 1]

# Use threshold stored in deployment configuration
threshold = config["threshold"]

test_predictions = (
    test_probabilities >= threshold
).astype(int)

# Calculate metrics
accuracy = accuracy_score(y_test, test_predictions)
precision = precision_score(y_test, test_predictions)
recall = recall_score(y_test, test_predictions)
f1 = f1_score(y_test, test_predictions)
roc_auc = roc_auc_score(y_test, test_probabilities)

cm = confusion_matrix(y_test, test_predictions)

print("SAVED MODEL VALIDATION")
print("=" * 45)

print(f"Threshold : {threshold}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

SAVED MODEL VALIDATION
Threshold : 0.5
Accuracy  : 0.8730
Precision : 0.8218
Recall    : 0.9823
F1 Score  : 0.8949
ROC-AUC   : 0.9831

Confusion Matrix:
[[27343  9657]
 [  803 44529]]


In [5]:
# Convert saved report into a dictionary
saved_metrics = dict(
    zip(
        final_report["Metric"],
        final_report["Value"]
    )
)

# Results reproduced by this fresh notebook
validation_metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc,
    "True Negatives": cm[0, 0],
    "False Positives": cm[0, 1],
    "False Negatives": cm[1, 0],
    "True Positives": cm[1, 1],
    "Decision Threshold": threshold
}

print("SAVED REPORT VALIDATION")
print("=" * 55)

all_metrics_match = True

for metric, value in validation_metrics.items():

    saved_value = saved_metrics[metric]

    match = np.isclose(
        value,
        saved_value,
        atol=0.0001
    )

    status = "MATCH" if match else "MISMATCH"

    print(f"{metric:20} : {status}")

    if not match:
        all_metrics_match = False

print("=" * 55)

if all_metrics_match:
    print("All saved metrics were successfully reproduced!")
else:
    print("WARNING: Some saved metrics do not match.")

SAVED REPORT VALIDATION
Accuracy             : MATCH
Precision            : MATCH
Recall               : MATCH
F1 Score             : MATCH
ROC-AUC              : MATCH
True Negatives       : MATCH
False Positives      : MATCH
False Negatives      : MATCH
True Positives       : MATCH
Decision Threshold   : MATCH
All saved metrics were successfully reproduced!


In [6]:
# Load original raw UNSW-NB15 test dataset

raw_test_path = data_dir / "raw" / "UNSW_NB15_testing-set.csv"

raw_test_df = pd.read_csv(raw_test_path)

# Select one attack record
raw_record = raw_test_df[
    raw_test_df["label"] == 1
].sample(n=1, random_state=42)

print("Raw record selected successfully!")

print("\nActual category:")
print(raw_record["attack_cat"].iloc[0])

print("\nActual label:")
print(raw_record["label"].iloc[0])

print("\nOriginal feature count:")
print(raw_record.shape[1])

Raw record selected successfully!

Actual category:
Backdoor

Actual label:
1

Original feature count:
45


In [7]:
# Remove columns that are not model input features
raw_features = raw_record.drop(
    columns=["id", "attack_cat", "label"],
    errors="ignore"
)

print("Raw model features:", raw_features.shape)

# Apply saved preprocessor
encoded_record = preprocessor.transform(raw_features)

print("After preprocessing:", encoded_record.shape)

# Apply saved scaler
scaled_record = scaler.transform(encoded_record)

print("After scaling:", scaled_record.shape)

# Verify model input size
print("Model expects:", model.n_features_in_, "features")

Raw model features: (1, 42)
After preprocessing: (1, 194)
After scaling: (1, 194)
Model expects: 194 features


In [8]:
# Predict attack probability
attack_probability = model.predict_proba(scaled_record)[0, 1]

# Get saved decision threshold
threshold = config["threshold"]

# Convert probability into final prediction
prediction = 1 if attack_probability >= threshold else 0

prediction_text = "ATTACK" if prediction == 1 else "NORMAL"

print("END-TO-END PREDICTION TEST")
print("=" * 40)

print("Actual category :", raw_record["attack_cat"].iloc[0])
print("Actual label    :", int(raw_record["label"].iloc[0]))

print()
print("Attack probability:", round(float(attack_probability), 4))
print("Decision threshold :", threshold)
print("Prediction         :", prediction_text)
print("Prediction code    :", prediction)

END-TO-END PREDICTION TEST
Actual category : Backdoor
Actual label    : 1

Attack probability: 0.9959
Decision threshold : 0.5
Prediction         : ATTACK
Prediction code    : 1


In [9]:
# Select one NORMAL record
normal_record = raw_test_df[
    raw_test_df["label"] == 0
].sample(n=1, random_state=42).copy()

# Prepare model features
normal_features = normal_record.drop(
    columns=["id", "attack_cat", "label"],
    errors="ignore"
)

# Preprocess and scale
normal_encoded = preprocessor.transform(normal_features)
normal_scaled = scaler.transform(normal_encoded)

# Predict probability
normal_probability = model.predict_proba(normal_scaled)[0, 1]

# Apply saved threshold
normal_prediction = (
    1 if normal_probability >= config["threshold"] else 0
)

normal_prediction_text = (
    "ATTACK" if normal_prediction == 1 else "NORMAL"
)

print("NORMAL TRAFFIC PREDICTION TEST")
print("=" * 40)

print("Actual category :", normal_record["attack_cat"].iloc[0])
print("Actual label    :", int(normal_record["label"].iloc[0]))

print()
print("Attack probability:", round(float(normal_probability), 4))
print("Decision threshold :", config["threshold"])
print("Prediction         :", normal_prediction_text)
print("Prediction code    :", normal_prediction)

NORMAL TRAFFIC PREDICTION TEST
Actual category : Normal
Actual label    : 0

Attack probability: 0.4004
Decision threshold : 0.5
Prediction         : NORMAL
Prediction code    : 0


In [10]:
print("=" * 60)
print("FINAL PROJECT VALIDATION SUMMARY")
print("=" * 60)

print("\nProject files:")
print("PASS - All required files found")

print("\nSaved pipeline:")
print("PASS - Model loaded")
print("PASS - Preprocessor loaded")
print("PASS - Scaler loaded")
print("PASS - Deployment configuration loaded")

print("\nModel reproducibility:")
print("PASS - Saved metrics reproduced successfully")

print("\nRaw attack test:")
print("Actual: Backdoor attack")
print("Prediction: ATTACK")
print("Attack probability: 99.59%")

print("\nRaw normal test:")
print("Actual: Normal traffic")
print("Prediction: NORMAL")
print("Attack probability: 40.04%")

print("\nFinal status:")
print("PROJECT VALIDATION PASSED")

print("=" * 60)

FINAL PROJECT VALIDATION SUMMARY

Project files:
PASS - All required files found

Saved pipeline:
PASS - Model loaded
PASS - Preprocessor loaded
PASS - Scaler loaded
PASS - Deployment configuration loaded

Model reproducibility:
PASS - Saved metrics reproduced successfully

Raw attack test:
Actual: Backdoor attack
Prediction: ATTACK
Attack probability: 99.59%

Raw normal test:
Actual: Normal traffic
Prediction: NORMAL
Attack probability: 40.04%

Final status:
PROJECT VALIDATION PASSED
